## Field-Level Comparison: Flat Values

**Purpose:** Line-by-line field comparison between the streaming and batch-load silver tables for catalog flat values.

| Role | Table |
|------|-------|
| Anchor (Stream) | `dev_sdsc_db.sdds_silver.catalog-stream-dbx-silver` |
| Compare (Load) | `dev_sdsc_db.sdds_silver.catalog-load-dbx-silver` |
| Target (Gold) | `dev_sdsc_db.sdds_gold.field_level_comparison_flat` |

**Logic:**
1. Full outer join on `partnumber`
2. For each flat field, compare values and produce:
   - `<field>_status` — MATCH / MISMATCH / MISSING_IN_STREAM / MISSING_IN_LOAD
   - `<field>_anchor_value` — value from stream table
   - `<field>_compare_value` — value from load table
   - `<field>_is_match` — boolean flag
3. `row_status` — indicates presence in both, stream-only, or load-only
4. Write results to gold table

In [0]:
#flat_values
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit, coalesce, row_number, expr
from pyspark.sql.types import ArrayType, MapType, StructType
from pyspark.sql.window import Window
from pyspark import StorageLevel
from datetime import datetime

# PERF: nothing was persisted before, so every .count() / .display() / write
# below re-scanned the source silver tables and re-ran the full-outer join +
# 200+ per-field similarity expressions from scratch. Persisting the two
# obvious checkpoints (post-read/dedup source frames, post-derivation joined
# frame) collapses that into one scan per side plus one join. DISK_ONLY:
# df_joined is ~200 * 4 = 800+ columns wide once every field has anchor/
# compare/is_match/similarity_pct — MEMORY_AND_DISK spills unpredictably at
# that width, disk is more predictable.
CACHE_LEVEL = StorageLevel.DISK_ONLY

# --- Configuration ---
STREAM_TABLE = "dev_sdsc_db.sdds_silver.`catalog-stream-dbx-silver`"
LOAD_TABLE = "dev_sdsc_db.sdds_silver.`catalog-load-dbx-silver`"
GOLD_TABLE = "dev_sdsc_db.sdds_gold.field_level_comparison_flat"

# --- Parameter: load_timestamp partition filter ---

dbutils.widgets.text("extraction_date", datetime.now().strftime("%Y-%m-%d"), "Extraction Date")

extraction_date = dbutils.widgets.get("extraction_date")
print(f"Running comparison for load_timestamp partition: {extraction_date}")

# Fields to compare (all columns except partnumber key and load_timestamp)
COMPARE_FIELDS = [    # Attribution
    'attributes', 'customSkuAttributes', 'defAttributes', 'floatFacets', 'numberFacets', 'searchAttributes', 'stringFacets',
    # Caddyshack
    'assetSeoUrl', 'catgroupSeq', 'dsgCatgroups', 'dsgSeoUrl', 'ggCatgroups', 'ggSeoUrl', 'leafCategories', 'parentCatgroup',
    'parentCatgroup0', 'parentCatgroup1', 'parentCatgroup2', 'parentCatgroup3', 'parentCatgroup4', 'parentCatgroup5',
    'parentCatgroup6', 'parentCatgroup7', 'parentCatgroup8', 'parentCatgroup9', 'plCatgroups', 'plSeoUrl',
    'primaryCategories_dsg_id', 'primaryCategories_dsg_identifier', 'primaryCategories_g3_id', 'primaryCategories_g3_identifier',
    'primaryCategories_gg_id', 'primaryCategories_gg_identifier', 'primaryCategories_pl_id', 'primaryCategories_pl_identifier',
    'productSearchFlag', 'seo',
    # MDM
    'auxDescription2', 'buyable', 'catalogIds', 'catentryId', 'color_family', 'color_seq', 'color_swatch',
    'comingSoonEndDateTime', 'dsgProductSortDate', 'dsgPublishOverride', 'endDate', 'endDateTime', 'extraction_date',
    'fullImage', 'ggAkamaiRedirect', 'ggKeywordOverride', 'ggProductSortDate', 'ggPublishOverride', 'ggUrl', 'keyword',
    'longDescription', 'mfName', 'name', 'onOrder', 'parentCatentryId', 'parentPartnumber', 'plProductSortDate',
    'plPublishOverride', 'primaryUPC', 'productGroup_id', 'productGroup_seq', 'productGroup_sequence', 'productType',
    'published', 'seo_asset_title', 'seo_dsg_imageAltDesc', 'seo_dsg_metaDesc', 'seo_dsg_metaKeyword', 'seo_dsg_title',
    'seo_gg_imageAltDesc', 'seo_gg_metaDesc', 'seo_gg_metaKeyword', 'seo_gg_title', 'seo_pl_imageAltDesc', 'seo_pl_metaDesc',
    'seo_pl_metaKeyword', 'seo_pl_title', 'startDate', 'startDateTime', 'swatchPartNumber', 'taxCode', 'thumbnail', 'type', 'webActiveDate',
    # Overrides
    'dsgOverrides_auxdescription1', 'dsgOverrides_auxdescription2', 'dsgOverrides_fullimage', 'dsgOverrides_longdescription',
    'dsgOverrides_name', 'dsgOverrides_published', 'dsgOverrides_thumbnail', 'ggOverrides_auxdescription1',
    'ggOverrides_auxdescription2', 'ggOverrides_fullimage', 'ggOverrides_longdescription', 'ggOverrides_name',
    'ggOverrides_published', 'ggOverrides_thumbnail', 'plOverrides_auxdescription1', 'plOverrides_auxdescription2',
    'plOverrides_fullimage', 'plOverrides_longdescription', 'plOverrides_name', 'plOverrides_published', 'plOverrides_thumbnail',
    # Pricing
    'dsgPriceIndicators_dealsPercentage', 'dsgPriceIndicators_mapPriceIndicator', 'dsgPriceIndicators_priceIndicator',
    'kafkaPriceList_identifier_dickssportinggoodslistprice_endDateTime', 'kafkaPriceList_identifier_dickssportinggoodslistprice_maxQty',
    'kafkaPriceList_identifier_dickssportinggoodslistprice_minQty', 'kafkaPriceList_identifier_dickssportinggoodslistprice_startDateTime',
    'kafkaPriceList_identifier_dickssportinggoodslistprice_stringValue', 'kafkaPriceList_identifier_dickssportinggoodslistprice_value',
    'kafkaPriceList_identifier_dickssportinggoodsmapprice_endDateTime', 'kafkaPriceList_identifier_dickssportinggoodsmapprice_maxQty',
    'kafkaPriceList_identifier_dickssportinggoodsmapprice_minQty', 'kafkaPriceList_identifier_dickssportinggoodsmapprice_startDateTime',
    'kafkaPriceList_identifier_dickssportinggoodsmapprice_stringValue', 'kafkaPriceList_identifier_dickssportinggoodsmapprice_value',
    'kafkaPriceList_identifier_dickssportinggoodsofferprice_endDateTime', 'kafkaPriceList_identifier_dickssportinggoodsofferprice_maxQty',
    'kafkaPriceList_identifier_dickssportinggoodsofferprice_minQty', 'kafkaPriceList_identifier_dickssportinggoodsofferprice_startDateTime',
    'kafkaPriceList_identifier_dickssportinggoodsofferprice_stringValue', 'kafkaPriceList_identifier_dickssportinggoodsofferprice_value',
    'kafkaPriceList_identifier_golfgalaxylistprice_endDateTime', 'kafkaPriceList_identifier_golfgalaxylistprice_maxQty',
    'kafkaPriceList_identifier_golfgalaxylistprice_minQty', 'kafkaPriceList_identifier_golfgalaxylistprice_startDateTime',
    'kafkaPriceList_identifier_golfgalaxylistprice_stringValue', 'kafkaPriceList_identifier_golfgalaxylistprice_value',
    'kafkaPriceList_identifier_golfgalaxymapprice_endDateTime', 'kafkaPriceList_identifier_golfgalaxymapprice_maxQty',
    'kafkaPriceList_identifier_golfgalaxymapprice_minQty', 'kafkaPriceList_identifier_golfgalaxymapprice_startDateTime',
    'kafkaPriceList_identifier_golfgalaxymapprice_stringValue', 'kafkaPriceList_identifier_golfgalaxymapprice_value',
    'kafkaPriceList_identifier_golfgalaxyofferprice_endDateTime', 'kafkaPriceList_identifier_golfgalaxyofferprice_maxQty',
    'kafkaPriceList_identifier_golfgalaxyofferprice_minQty', 'kafkaPriceList_identifier_golfgalaxyofferprice_startDateTime',
    'kafkaPriceList_identifier_golfgalaxyofferprice_stringValue', 'kafkaPriceList_identifier_golfgalaxyofferprice_value',
    'kafkaPriceList_identifier_publiclandslistprice_endDateTime', 'kafkaPriceList_identifier_publiclandslistprice_maxQty',
    'kafkaPriceList_identifier_publiclandslistprice_minQty', 'kafkaPriceList_identifier_publiclandslistprice_startDateTime',
    'kafkaPriceList_identifier_publiclandslistprice_stringValue', 'kafkaPriceList_identifier_publiclandslistprice_value',
    'kafkaPriceList_identifier_publiclandsmapprice_endDateTime', 'kafkaPriceList_identifier_publiclandsmapprice_maxQty',
    'kafkaPriceList_identifier_publiclandsmapprice_minQty', 'kafkaPriceList_identifier_publiclandsmapprice_startDateTime',
    'kafkaPriceList_identifier_publiclandsmapprice_stringValue', 'kafkaPriceList_identifier_publiclandsmapprice_value',
    'kafkaPriceList_identifier_publiclandsofferprice_endDateTime', 'kafkaPriceList_identifier_publiclandsofferprice_maxQty',
    'kafkaPriceList_identifier_publiclandsofferprice_minQty', 'kafkaPriceList_identifier_publiclandsofferprice_startDateTime',
    'kafkaPriceList_identifier_publiclandsofferprice_stringValue', 'kafkaPriceList_identifier_publiclandsofferprice_value',
    'plPriceIndicators_dealsPercentage', 'plPriceIndicators_mapPriceIndicator', 'plPriceIndicators_priceIndicator',
    # Ranking
    'ranking_dsg_brand', 'ranking_dsg_cheap', 'ranking_dsg_clearance', 'ranking_dsg_cost', 'ranking_dsg_expensive',
    'ranking_dsg_newness', 'ranking_dsg_ratings', 'ranking_dsg_sale', 'ranking_dsg_salesDollars', 'ranking_dsg_salesMargin',
    'ranking_dsg_salesProfit', 'ranking_dsg_salesQuantity', 'ranking_dsg_salesTotalDollars', 'ranking_dsg_salesTotalMargin',
    'ranking_dsg_salesTotalProfit', 'ranking_dsg_salesTotalQuantity', 'ranking_dsg_savings', 'ranking_dsg_verticalBrand',
    'ranking_gg_brand', 'ranking_gg_cheap', 'ranking_gg_clearance', 'ranking_gg_cost', 'ranking_gg_expensive',
    'ranking_gg_newness', 'ranking_gg_ratings', 'ranking_gg_sale', 'ranking_gg_salesDollars', 'ranking_gg_salesMargin',
    'ranking_gg_salesProfit', 'ranking_gg_salesQuantity', 'ranking_gg_salesTotalDollars', 'ranking_gg_salesTotalMargin',
    'ranking_gg_salesTotalProfit', 'ranking_gg_salesTotalQuantity', 'ranking_gg_savings', 'ranking_gg_verticalBrand',
    'ranking_pl_brand', 'ranking_pl_cheap', 'ranking_pl_clearance', 'ranking_pl_cost', 'ranking_pl_expensive',
    'ranking_pl_newness', 'ranking_pl_ratings', 'ranking_pl_sale', 'ranking_pl_salesDollars', 'ranking_pl_salesMargin',
    'ranking_pl_salesProfit', 'ranking_pl_salesQuantity', 'ranking_pl_salesTotalDollars', 'ranking_pl_salesTotalMargin',
    'ranking_pl_salesTotalProfit', 'ranking_pl_salesTotalQuantity', 'ranking_pl_savings', 'ranking_pl_verticalBrand',
    # Sales
    'dsgQuantitySold', 'dsgTotalPriceSold', 'ggQuantitySold', 'ggTotalPriceSold', 'plQuantitySold', 'plTotalPriceSold',
    'salesData_ecomm_10701_costDollars', 'salesData_ecomm_10701_margin', 'salesData_ecomm_10701_qtySold', 'salesData_ecomm_10701_salesDollars',
    'salesData_ecomm_11201_costDollars', 'salesData_ecomm_11201_margin', 'salesData_ecomm_11201_qtySold', 'salesData_ecomm_11201_salesDollars',
    'salesData_ecomm_15108_costDollars', 'salesData_ecomm_15108_margin', 'salesData_ecomm_15108_qtySold', 'salesData_ecomm_15108_salesDollars',
    'salesData_ecomm_16066_costDollars', 'salesData_ecomm_16066_margin', 'salesData_ecomm_16066_qtySold', 'salesData_ecomm_16066_salesDollars',
    'salesData_total_10701_costDollars', 'salesData_total_10701_margin', 'salesData_total_10701_qtySold', 'salesData_total_10701_salesDollars',
    'salesData_total_11201_costDollars', 'salesData_total_11201_margin', 'salesData_total_11201_qtySold', 'salesData_total_11201_salesDollars',
    'salesData_total_15108_costDollars', 'salesData_total_15108_margin', 'salesData_total_15108_qtySold', 'salesData_total_15108_salesDollars',
    'salesData_total_16066_costDollars', 'salesData_total_16066_margin', 'salesData_total_16066_qtySold', 'salesData_total_16066_salesDollars',
    # Web Active Flag
    'caliaWebActive', 'dsgAppWebActive', 'dsgMobileAppWebActive', 'dsgWebActive', 'g3WebActive', 'ggAppWebActive',
    'ggMobileAppWebActive', 'ggWebActive', 'plWebActive', 'stackdWebActive', 'vrstWebActive',"load_timestamp"
]

# Long free-text / descriptive fields — get word-set Jaccard similarity instead
# of scalar 100/0. A long description that differs by a word or two would
# otherwise register as 0% similarity even though most of the content agrees.
TEXT_SIMILARITY_FIELDS = {
    'longDescription',
    'name', 'mfName', 'keyword',
    'searchAttributes',
    'auxDescription2',
    'dsgOverrides_name', 'ggOverrides_name', 'plOverrides_name',
    'dsgOverrides_longdescription', 'ggOverrides_longdescription', 'plOverrides_longdescription',
    'dsgOverrides_auxdescription1', 'dsgOverrides_auxdescription2',
    'ggOverrides_auxdescription1', 'ggOverrides_auxdescription2',
    'plOverrides_auxdescription1', 'plOverrides_auxdescription2',
    'seo_asset_title',
    'seo_dsg_title', 'seo_dsg_metaDesc', 'seo_dsg_metaKeyword', 'seo_dsg_imageAltDesc',
    'seo_gg_title', 'seo_gg_metaDesc', 'seo_gg_metaKeyword', 'seo_gg_imageAltDesc',
    'seo_pl_title', 'seo_pl_metaDesc', 'seo_pl_metaKeyword', 'seo_pl_imageAltDesc',
    'ggKeywordOverride',
}

# ── Per-row similarity (Jaccard-based, 0-100) ──────────────────────────────
# Ported from run_id.py: every field gets a similarity_pct so "mostly the
# same" rows score higher than "completely disjoint" rows instead of both
# collapsing into a MISMATCH bucket.
#   - scalar (int/bool/string/date/timestamp/struct): 100 if `<=>` else 0.
#     Null-safe: both null => 100, one null one not => 0.
#   - array<scalar>: |A ∩ B| / |A ∪ B| * 100.
#   - array<struct|map|array>: same, but each element cast to string first
#     so array_intersect/array_union can compare them.
#   - map<...>: flatten each key's value list into "key||value" pairs, then
#     Jaccard over pairs (drift in value shows up as fully disjoint pairs;
#     extra values under a shared key reduce similarity smoothly).
#   - text (long free-text): word-set Jaccard on the raw column
#     (lowercased, whitespace-split).
def _similarity_core(a_ref, c_ref, shape):
    if shape == "scalar":
        return f"CASE WHEN {a_ref} <=> {c_ref} THEN 100.0 ELSE 0.0 END"

    if shape == "array":
        return f"""
            CASE
              WHEN {a_ref} IS NULL AND {c_ref} IS NULL THEN 100.0
              WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0.0
              WHEN size(array_union({a_ref}, {c_ref})) = 0 THEN 100.0
              ELSE size(array_intersect({a_ref}, {c_ref})) * 100.0 / size(array_union({a_ref}, {c_ref}))
            END
        """

    if shape == "array_struct":
        a_strs = f"transform(coalesce({a_ref}, array()), s -> cast(s as string))"
        c_strs = f"transform(coalesce({c_ref}, array()), s -> cast(s as string))"
        return f"""
            CASE
              WHEN {a_ref} IS NULL AND {c_ref} IS NULL THEN 100.0
              WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0.0
              WHEN size(array_union({a_strs}, {c_strs})) = 0 THEN 100.0
              ELSE size(array_intersect({a_strs}, {c_strs})) * 100.0 / size(array_union({a_strs}, {c_strs}))
            END
        """

    if shape == "map":
        # Native map<K,V>. Flatten each key's value list to "key||value" pairs,
        # then Jaccard over pairs (drift on value shows as disjoint pairs; an
        # extra value under a shared key reduces similarity smoothly).
        a_pairs = (
            f"transform(map_entries(coalesce({a_ref}, map())), "
            f"e -> concat_ws('||', cast(e.key as string), cast(e.value as string)))"
        )
        c_pairs = (
            f"transform(map_entries(coalesce({c_ref}, map())), "
            f"e -> concat_ws('||', cast(e.key as string), cast(e.value as string)))"
        )
        return f"""
            CASE
              WHEN {a_ref} IS NULL AND {c_ref} IS NULL THEN 100.0
              WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0.0
              WHEN size(array_union({a_pairs}, {c_pairs})) = 0 THEN 100.0
              ELSE size(array_intersect({a_pairs}, {c_pairs})) * 100.0 / size(array_union({a_pairs}, {c_pairs}))
            END
        """

    if shape == "array_map":
        # array<map<K,V>> (e.g. `attributes`). flatten map_entries across every
        # map element, THEN Jaccard over pairs. Kept as a separate shape from
        # "map" because a runtime CASE WHEN over `typeof(...)` fails compile-
        # time type checking (both arms of CASE must resolve to the same type,
        # so wrapping a native map in `array(map)` produced array<array<map>>
        # vs array<map> and Spark rejected it as DATATYPE_MISMATCH). Deciding
        # the shape in Python from the schema lets each branch emit a single,
        # already-typed expression.
        a_pairs = (
            f"flatten(transform(coalesce({a_ref}, array()), "
            f"m -> transform(map_entries(coalesce(m, map())), "
            f"e -> concat_ws('||', cast(e.key as string), cast(e.value as string)))))"
        )
        c_pairs = (
            f"flatten(transform(coalesce({c_ref}, array()), "
            f"m -> transform(map_entries(coalesce(m, map())), "
            f"e -> concat_ws('||', cast(e.key as string), cast(e.value as string)))))"
        )
        return f"""
            CASE
              WHEN {a_ref} IS NULL AND {c_ref} IS NULL THEN 100.0
              WHEN {a_ref} IS NULL OR {c_ref} IS NULL THEN 0.0
              WHEN size(array_union({a_pairs}, {c_pairs})) = 0 THEN 100.0
              ELSE size(array_intersect({a_pairs}, {c_pairs})) * 100.0 / size(array_union({a_pairs}, {c_pairs}))
            END
        """

    if shape == "text":
        a_words = f"filter(array_distinct(split(lower(coalesce({a_ref}, '')), '\\\\s+')), w -> w != '')"
        c_words = f"filter(array_distinct(split(lower(coalesce({c_ref}, '')), '\\\\s+')), w -> w != '')"
        return f"""
            CASE
              WHEN size(array_union({a_words}, {c_words})) = 0 THEN 100.0
              ELSE size(array_intersect({a_words}, {c_words})) * 100.0 / size(array_union({a_words}, {c_words}))
            END
        """

    raise ValueError(f"Unknown similarity shape: {shape}")


def build_similarity_expr(a_ref, c_ref, shape):
    """Returns a DOUBLE column (0.0 - 100.0) rounded to 2dp."""
    return expr(f"cast(round({_similarity_core(a_ref, c_ref, shape)}, 2) as double)")


def _shape_for(df_joined, field):
    """Infer similarity shape from the joined-DF schema of the `s_` prefixed column.

    The stream side is prefix-aliased only (types preserved), so the schema
    lookup returns the original data type. TEXT_SIMILARITY_FIELDS overrides
    schema-based inference for long free-text columns since we can't tell
    "short identifier string" from "long descriptive string" by type alone.
    """
    if field in TEXT_SIMILARITY_FIELDS:
        return "text"
    prefixed = f"s_{field}"
    if prefixed not in df_joined.columns:
        return "scalar"
    dt = df_joined.schema[prefixed].dataType

    if isinstance(dt, MapType):
        return "map"
    if isinstance(dt, ArrayType):
        elem = dt.elementType
        if isinstance(elem, MapType):
            # array<map<...>> — needs its own shape ("array_map") because the
            # SQL to flatten map_entries across an array of maps has different
            # types (and one more transform level) than for a native map<...>.
            return "array_map"
        if isinstance(elem, (StructType, ArrayType)):
            return "array_struct"
        return "array"
    # scalar / struct / everything else — <=> equality
    return "scalar"


# --- Read source tables filtered by load_timestamp partition ---
df_stream_raw = (
    spark.table(STREAM_TABLE)
    .filter(col("load_timestamp").cast("date") == extraction_date)
)
# Deduplicate stream by partnumber, keeping the latest record
_w = Window.partitionBy("partnumber", "parentPartnumber").orderBy(col("load_timestamp").desc())
df_stream = (
    df_stream_raw
    .withColumn("_rn", row_number().over(_w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
df_load = (
    spark.table(LOAD_TABLE)
    .filter(col("load_timestamp").cast("date") == extraction_date)
)

# PERF: cache the two source frames once. Each is used at least three times
# below (raw count for the print line, prefixed alias for the join, then the
# join itself), and without a persist the .count() and the join both re-scan
# the silver table + re-run the dedup Window. One materialize per side ->
# one scan per side.
df_stream_raw = df_stream_raw.persist(CACHE_LEVEL)
df_stream = df_stream.persist(CACHE_LEVEL)
df_load = df_load.persist(CACHE_LEVEL)

print(f"Stream rows after dedup: {df_stream.count():,}  (raw: {df_stream_raw.count():,})")
print(f"Load rows: {df_load.count():,}")

df_stream_prefixed = df_stream.select(
    [col(c).alias(f"s_{c}") for c in df_stream.columns]
)
df_load_prefixed = df_load.select(
    [col(c).alias(f"l_{c}") for c in df_load.columns]
)

# --- Full outer join on composite (partnumber, parentPartnumber) ---
# Composite key mirrors attribute_comp.py's make_match_key: the same
# partnumber under two different parents on the two sides is now treated
# as two unrelated records, not force-paired by partnumber alone.
# eqNullSafe on parentPartnumber so top-level records that legitimately
# carry NULL on both sides still align (Spark's plain "=" treats NULL !=
# NULL and would drop them to MISSING_IN_* otherwise).
df_joined = df_stream_prefixed.join(
    df_load_prefixed,
    on=(
        (col("s_partnumber") == col("l_partnumber"))
        & col("s_parentPartnumber").eqNullSafe(col("l_parentPartnumber"))
    ),
    how="full_outer",
)

# Resolve partnumber and parentPartnumber from both sides
df_joined = df_joined.withColumn(
    "partnumber", coalesce(col("s_partnumber"), col("l_partnumber"))
).withColumn(
    "parentPartnumber", coalesce(col("s_parentPartnumber"), col("l_parentPartnumber"))
)

# --- Row-level status ---
df_joined = df_joined.withColumn(
    "row_status",
    when(col("s_partnumber").isNull(), lit("MISSING_IN_STREAM"))
    .when(col("l_partnumber").isNull(), lit("MISSING_IN_LOAD"))
    .otherwise(lit("PRESENT_IN_BOTH")),
)

# --- Field-level comparison (batch all columns at once for performance) ---
new_columns = {}
for field in COMPARE_FIELDS:
    s_col = col(f"s_{field}")
    l_col = col(f"l_{field}")

    # anchor_value (stream) and compare_value (load)
    new_columns[f"{field}_anchor_value"] = s_col.cast("string")
    new_columns[f"{field}_compare_value"] = l_col.cast("string")

    # is_match: true when both null or both equal
    new_columns[f"{field}_is_match"] = (
        when((s_col.isNull()) & (l_col.isNull()), lit(True))
        .when(s_col.cast("string") == l_col.cast("string"), lit(True))
        .otherwise(lit(False))
    )

df_joined = df_joined.withColumns(new_columns)

# ── Document-availability gate ─────────────────────────────────────────────
# A row where either side's `type` is NULL is treated as an "unavailable
# document": the partnumber joined but at least one side never populated the
# document type, so every field on that row is being compared against a hole
# and any similarity number would skew both per-field and row-level averages.
# Flag it here (visible per-row for audit) and force similarity_pct /
# row_similarity_pct to NULL further down so this row does not contribute to
# the report. type_anchor_value / type_compare_value already carry the raw
# stream/load `type` values after the first pass above.
df_joined = df_joined.withColumn(
    "document_available",
    col("type_anchor_value").isNotNull() & col("type_compare_value").isNotNull(),
)

# --- Field status + similarity_pct (depend on row_status / is_match,
#     so applied in a second pass so is_match has already materialized) ---
# similarity_pct is NULL when the row is missing on one side OR the document
# is flagged unavailable — either case there is nothing meaningful to compare
# against, and letting these rows resolve to 0.0 would drag every field-level
# average toward zero for reasons that are not real drift.
second_pass = {}
for field in COMPARE_FIELDS:
    shape = _shape_for(df_joined, field)
    sim_expr = build_similarity_expr(f"s_{field}", f"l_{field}", shape)

    second_pass[f"{field}_status"] = (
        when(col("row_status") == "MISSING_IN_STREAM", lit("MISSING_IN_STREAM"))
        .when(col("row_status") == "MISSING_IN_LOAD", lit("MISSING_IN_LOAD"))
        .when(col(f"{field}_is_match") == True, lit("MATCH"))
        .otherwise(lit("MISMATCH"))
    )
    second_pass[f"{field}_similarity_pct"] = (
        when(col("row_status") != "PRESENT_IN_BOTH", lit(None).cast("double"))
        .when(~col("document_available"), lit(None).cast("double"))
        .otherwise(sim_expr)
    )

df_joined = df_joined.withColumns(second_pass)

# --- Row-level similarity: mean of per-field similarities (nulls skipped) ---
_sim_array = "array(" + ", ".join([f"`{f}_similarity_pct`" for f in COMPARE_FIELDS]) + ")"
_non_null = f"filter({_sim_array}, x -> x is not null)"
df_joined = df_joined.withColumn(
    "row_similarity_pct",
    when(col("row_status") != "PRESENT_IN_BOTH", lit(None).cast("double"))
    .when(~col("document_available"), lit(None).cast("double"))
    .otherwise(
        expr(
            f"cast(round("
            f"  aggregate({_non_null}, cast(0.0 as double), (acc, x) -> acc + x) "
            f"  / greatest(size({_non_null}), 1), "
            f"2) as double)"
        )
    ),
)

# --- Select final columns in gold table order ---
status_cols = [f"{f}_status" for f in COMPARE_FIELDS]
detail_cols = []
for f in COMPARE_FIELDS:
    detail_cols.extend([
        f"{f}_anchor_value",
        f"{f}_compare_value",
        f"{f}_is_match",
        f"{f}_similarity_pct",
    ])

final_columns = (
    ["partnumber", "row_status", "document_available", "row_similarity_pct", "type_stream", "type_load"]
    + status_cols
    + detail_cols
)

# partnumber_status mirrors row_status since it's the join key
df_joined = df_joined.withColumn(
    "partnumber_status",
    when(col("row_status") == "PRESENT_IN_BOTH", lit("MATCH"))
    .when(col("row_status") == "MISSING_IN_STREAM", lit("MISSING_IN_STREAM"))
    .otherwise(lit("MISSING_IN_LOAD")),
)

# Bring raw type values from each source table
df_joined = df_joined.withColumn("type_stream", col("s_type").cast("string"))
df_joined = df_joined.withColumn("type_load", col("l_type").cast("string"))

# Add load_timestamp column for partitioning
df_result = df_joined.select(final_columns).withColumn(
    "extraction_date", lit(extraction_date).cast("timestamp")
)

# PERF: df_result is the widest, most expensive frame in the notebook
# (200+ per-field similarity expressions on top of a full-outer join over
# two silver tables). Every downstream action — the .count() print below,
# the display() call, and whatever write step consumes it next — would
# otherwise re-execute the entire lineage. Persist once here so all three
# reuse the same materialized frame.
df_result = df_result.persist(CACHE_LEVEL)

print(f"Total rows in comparison: {df_result.count()}")


Running comparison for load_timestamp partition: 2026-09-09
Stream rows after dedup: 1,812,825  (raw: 1,812,825)
Load rows: 2,913,192
Total rows in comparison: 2977328


partnumber row_status row_similarity_pct type_stream type_load attributes_status customSkuAttributes_status defAttributes_status floatFacets_status numberFacets_status searchAttributes_status stringFacets_status assetSeoUrl_status catgroupSeq_status dsgCatgroups_status dsgSeoUrl_status ggCatgroups_status ggSeoUrl_status leafCategories_status parentCatgroup_status parentCatgroup0_status parentCatgroup1_status parentCatgroup2_status parentCatgroup3_status parentCatgroup4_status parentCatgroup5_status parentCatgroup6_status parentCatgroup7_status parentCatgroup8_status parentCatgroup9_status plCatgroups_status plSeoUrl_status primaryCategories_dsg_id_status primaryCategories_dsg_identifier_status primaryCategories_g3_id_status primaryCategories_g3_identifier_status primaryCategories_gg_id_status primaryCategories_gg_identifier_status primaryCategories_pl_id_status primaryCategories_pl_identifier_status productSearchFlag_status seo_status auxDescription2_status buyable_status catalogIds_status catentryId_status color_family_status color_seq_status color_swatch_status comingSoonEndDateTime_status dsgProductSortDate_status dsgPublishOverride_status endDate_status endDateTime_status extraction_date_status fullImage_status ggAkamaiRedirect_status ggKeywordOverride_status ggProductSortDate_status ggPublishOverride_status ggUrl_status keyword_status longDescription_status mfName_status name_status onOrder_status parentCatentryId_status parentPartnumber_status plProductSortDate_status plPublishOverride_status primaryUPC_status productGroup_id_status productGroup_seq_status productGroup_sequence_status productType_status published_status seo_asset_title_status seo_dsg_imageAltDesc_status seo_dsg_metaDesc_status seo_dsg_metaKeyword_status seo_dsg_title_status seo_gg_imageAltDesc_status seo_gg_metaDesc_status seo_gg_metaKeyword_status seo_gg_title_status seo_pl_imageAltDesc_status seo_pl_metaDesc_status seo_pl_metaKeyword_status seo_pl_title_status startDate_status startDateTime_status swatchPartNumber_status taxCode_status thumbnail_status type_status webActiveDate_status dsgOverrides_auxdescription1_status dsgOverrides_auxdescription2_status dsgOverrides_fullimage_status dsgOverrides_longdescription_status dsgOverrides_name_status dsgOverrides_published_status dsgOverrides_thumbnail_status ggOverrides_auxdescription1_status ggOverrides_auxdescription2_status ggOverrides_fullimage_status ggOverrides_longdescription_status ggOverrides_name_status ggOverrides_published_status ggOverrides_thumbnail_status plOverrides_auxdescription1_status plOverrides_auxdescription2_status plOverrides_fullimage_status plOverrides_longdescription_status plOverrides_name_status plOverrides_published_status plOverrides_thumbnail_status dsgPriceIndicators_dealsPercentage_status dsgPriceIndicators_mapPriceIndicator_status dsgPriceIndicators_priceIndicator_status kafkaPriceList_identifier_dickssportinggoodslistprice_endDateTime_status kafkaPriceList_identifier_dickssportinggoodslistprice_maxQty_status kafkaPriceList_identifier_dickssportinggoodslistprice_minQty_status kafkaPriceList_identifier_dickssportinggoodslistprice_startDateTime_status kafkaPriceList_identifier_dickssportinggoodslistprice_stringValue_status kafkaPriceList_identifier_dickssportinggoodslistprice_value_status kafkaPriceList_identifier_dickssportinggoodsmapprice_endDateTime_status kafkaPriceList_identifier_dickssportinggoodsmapprice_maxQty_status kafkaPriceList_identifier_dickssportinggoodsmapprice_minQty_status kafkaPriceList_identifier_dickssportinggoodsmapprice_startDateTime_status kafkaPriceList_identifier_dickssportinggoodsmapprice_stringValue_status kafkaPriceList_identifier_dickssportinggoodsmapprice_value_status kafkaPriceList_identifier_dickssportinggoodsofferprice_endDateTime_status kafkaPriceList_identifier_dickssportinggoodsofferprice_maxQty_status kafkaPriceList_identifier_dickssportinggoodsofferprice_minQty_status kafkaPriceList_identifier_dickssportinggoodsofferprice_startDateTime

In [0]:
# Write comparison results to the gold table (append, partitioned by extraction_date)
# If the table exists but was partitioned by the old column, drop and recreate
#spark.sql(f"DROP TABLE {GOLD_TABLE}")

if spark.catalog.tableExists(GOLD_TABLE):
    existing_partitions = spark.sql(f"DESCRIBE DETAIL {GOLD_TABLE}").select("partitionColumns").first()[0]
    
if not spark.catalog.tableExists(GOLD_TABLE):
    # First run: create table with extraction_date partition
    (
        df_result.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("extraction_date")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_TABLE)
    )
else:
    # Subsequent runs: overwrite only this partition (idempotent re-runs)
    (
        df_result.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"extraction_date = '{extraction_date}'")
        .partitionBy("extraction_date")
        .option("mergeSchema", "true")
        .saveAsTable(GOLD_TABLE)
    )

print(f"Successfully wrote to {GOLD_TABLE} (partition: extraction_date = '{extraction_date}')")
print(f"Total row count in table: {spark.table(GOLD_TABLE).count()}")

Successfully wrote to dev_sdsc_db.sdds_gold.field_level_comparison_flat (partition: extraction_date = '2026-09-09')
Total row count in table: 41247199
